# 03 — Energy-aware A* and the cost matrix

A* combines the energy already spent (`g`) with an optimistic remaining-energy estimate (`h`). Its path cost should equal UCS while generally expanding fewer cells. The cost matrix then runs A* for every directed pair of stops.


In [3]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from numpy.random import default_rng
from uavplanner.config import Config
from uavplanner.informed import astar, cost_matrix
from uavplanner.models import EnergyCoef
from uavplanner.uninformed import ucs
from uavplanner.viz import write_figures2
from uavplanner.world import make_targets, make_world

cfg = Config()
rng = default_rng(cfg.seed)
world = make_world(cfg, rng)
targets = make_targets(world, cfg, rng)
coef = EnergyCoef()
stops = [world.base] + [target.pos for target in targets[:3]]
start, goal = stops[0], stops[1]

ucs_result = ucs(world, world.no_fly, start, goal, coef, payload=1.0)
astar_result = astar(world, world.no_fly, start, goal, coef, payload=1.0, max_wind=cfg.wind_limit)
print(f'UCS:  cost={ucs_result.cost:.2f} EU, expanded={ucs_result.expanded}')
print(f'A*:   cost={astar_result.cost:.2f} EU, expanded={astar_result.expanded}')

matrix, paths = cost_matrix(world, world.no_fly, coef, stops, payload=1.0, max_wind=cfg.wind_limit)
print('Directed energy matrix (EU):')
print(matrix)

UCS:  cost=13.68 EU, expanded=212
A*:   cost=13.68 EU, expanded=168
Directed energy matrix (EU):
[[ 0.         13.67711793 50.73947131 47.97686394]
 [15.3814088   0.         37.06235338 34.49175283]
 [53.78310292 38.40169412  0.          8.05920849]
 [65.44096134 50.25141279 22.47959702  0.        ]]


In [4]:
world_png, explored_png, comparison_png = write_figures2(ROOT / 'results')
print(f'Wrote {explored_png}')
print(f'Wrote {comparison_png}')

Wrote /media/abdullahmughal/b74a1ef7-556e-41b4-9ad8-72e96d04151d/Energy_Aware_UAV_Mission_Planner/results/fig_search_explored.png
Wrote /media/abdullahmughal/b74a1ef7-556e-41b4-9ad8-72e96d04151d/Energy_Aware_UAV_Mission_Planner/results/fig_search_compare.png
